In [1]:
from run_inference import *

args = argparse.Namespace(expt_id="C26111221", device=0, old=False, dataset="cifar")

experiment_id = args.expt_id
folder = "models" if not args.old else "models_old"
expt_root = f"{folder}/{experiment_id}/"

dataset = args.dataset
DEVICE = f"cuda:{args.device}" if torch.cuda.is_available() and args.device != -1 else "cpu"

import pickle
with open(os.path.join(expt_root, "args.pkl"), "rb") as file:
    args = pickle.load(file)

model, scoremodel, embed_model, preembed_model, aggregator = load_models(name="best", expt_root=expt_root, device=DEVICE, args=args)
model.eval(), scoremodel.eval(), embed_model.eval(), preembed_model.eval()
if aggregator is not None:
    aggregator.eval()

nwt, lamwt = next(scoremodel.parameters()).exp().detach().cpu()
nwt, lamwt = nwt.item(), lamwt.item()

print(f"normscore weight: {nwt:.4f}, lamscore weight: {lamwt:.4f}")

DATA_ROOT = f"final_data/{dataset}"
TRAIN_FILE = f"{DATA_ROOT}/dataset_train.hdf5"
VAL_FILE = f"{DATA_ROOT}/dataset_val.hdf5"
TEST_FILE = f"{DATA_ROOT}/dataset_test.hdf5"

image_embed_model, TRAIN_FILE, VAL_FILE, TEST_FILE = get_image_embed_model(dataset, [TRAIN_FILE, VAL_FILE, TEST_FILE], device=DEVICE)

test_dataset = PairDatasetTest(TEST_FILE)

##### HPARAMS ######
b = args.b          # hinge margin for negative gap penalty (b-Apa)
b1 = args.b1         # hinge margin for positive gap penalty (Apa-b)
delta = args.delta     # hinge margin for contrastive loss
lr = args.lr
nepochs = args.nepochs
stagger = args.stagger
lamwt = args.lamwt    # loss coefficient for negative gap penalty
gapwt = args.gapwt       # loss coefficient for positive gap penalty
M = test_dataset.q[0].shape[0]
N = test_dataset.c[0].shape[0]
####################

A_mat, a_vec, Rm_mat = get_opas_constants(M, N, DEVICE)

CFG = AttributeDict({
    'tau': 1,
    'n_sink_iter': 20,
    'n_samples': 1,
})

DEEPSET = False #args.deepset
NOLAMMODEL = False #args.no_lammodel   
positive_samples = 10

Loading `best` model
normscore weight: 1.0515, lamscore weight: 1.6433


In [59]:
import itertools

PERMS = torch.tensor(list(itertools.permutations(range(6)))).long()

In [60]:
q_ = torch.randn(6, 4000)
num_samples = 10
torch.manual_seed(0)
perms = torch.stack([torch.arange(len(q_))] + [s for s in [torch.randperm(len(q_),) for _ in range(num_samples)] if not torch.equal(s, torch.arange(len(q_)))]).long()

In [105]:
import itertools


@torch.no_grad()
def compute_odr(dataset, model, scoremodel, embed_model, preembed_model, image_embed_model=None, n_samp=20, stagger=2, verbose=False):

    loader = dataset.get_dataloader(batch_size=1, shuffle=True)

    num_samples = n_samp
    C = embed_full_corpus(dataset, embed_model, preembed_model, image_embed_model=image_embed_model)
    # C is (N, n, xoutdim)
    M = dataset.q[0].shape[0]
    PERMS = torch.tensor(list(itertools.permutations(range(M)))).long()[1:]

    queries = []
    corps = []
    labels = []

    for n, (q, l) in enumerate(tqdm(loader, leave=True, disable=not verbose)):
        if n > 5: break
        q_, l_ = q[0], l[0]

        # shuffles = torch.stack([torch.arange(len(q_))] + [s for s in [torch.randperm(len(q_),) for _ in range(num_samples)] if not torch.equal(s, torch.arange(len(q_)))])
        shuffles = torch.vstack((torch.arange(len(q_))[None], PERMS[torch.randperm(PERMS.shape[0])[:num_samples]])) 

        q = q_[shuffles]
        l = l_[None].repeat_interleave(len(q_), dim=0)
        true_c = C[torch.where(l[0])[0]]
        print(q.shape, l.shape, true_c.shape)

        queries.append(q)
        corps.append(true_c)
        labels.append(l)

    queries = torch.vstack(queries)
    corps = torch.vstack(corps)
    labels = torch.vstack(labels)

    print(queries.shape, corps.shape, labels.shape)
    raise

    odc = []
    for i in range(queries.shape[0]):

        q = q.to(next(model.parameters()).device) 
        # q is bmd, c is Nnd, l is bN
        q = q + batch_get_white_noise(q, args.SNR)     # torch.randn_like(q) * noise
        q = embed_if_image_and_normalize(q, image_embed_model)
        q = embed_model(preembed_model(q))
        q = normalize(q)
        # q is (b, m, xoutdim)

        qct = torch.einsum("bmd,Nnd->bNmn", q, true_c)  # verified
        model_inputs = stagger_and_concat(qct, num_stagger=stagger) # bNsmn  s = num_stagger+1

        lambdas = torch.stack([model(x) for x in model_inputs])
        # bNm1

        F_mat = Rm_mat.T @ (2*qct + (a_vec @ lambdas.transpose(2,3) @ A_mat).transpose(2,3))

        P = gumbel_sinkhorn(F_mat, CFG.tau, CFG.n_sink_iter, noise=False)

        RmPC = Rm_mat @ P @ true_c.squeeze(-1)

        lamscore = lamwt * (lambdas.transpose(2,3) @ F.relu(b-A_mat @ Rm_mat @ P @ a_vec)).squeeze()
        normscore = torch.norm(q.unsqueeze(1) - RmPC, dim=[-1,-2])

        allscores = torch.stack([lamscore, normscore], dim=2)
        netscore = 2*scoremodel(-allscores).squeeze()      # b
        
        pos = netscore[0]
        neg = netscore[1:]
        diff = (pos - neg)

        odc.append((100 * (pos-neg > 0).sum() / np.prod(diff.shape)).item())

    return np.mean(odc)

In [106]:
compute_odr(test_dataset, model, scoremodel, embed_model, preembed_model, image_embed_model, stagger=args.stagger, verbose=True, n_samp=20)

  0%|                                                           | 3/12000 [00:01<1:04:14,  3.11it/s]

torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])


  0%|                                                             | 8/12000 [00:01<21:34,  9.26it/s]

torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])


  0%|                                                            | 15/12000 [00:01<11:04, 18.04it/s]

...

  1%|▍                                                           | 95/12000 [00:04<06:29, 30.60it/s]

torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size([10, 20, 32])
torch.Size([21, 6, 3, 32, 32]) torch.Size([6, 6000]) torch.Size(

  1%|▍                                                          | 101/12000 [00:04<09:24, 21.07it/s]


torch.Size([2121, 6, 3, 32, 32]) torch.Size([1010, 20, 32]) torch.Size([606, 6000])


RuntimeError: No active exception to reraise

In [ ]:
def get_mean_std(arr):
    return np.mean(arr), np.std(arr)

def get_mean_std_formatted(arr):
    mu, sigma = get_mean_std(arr)
    return f"{mu:.4f}±{sigma:.4f}"

In [32]:
odc = []
for _ in range(10):
    odc_ = compute_odr(test_dataset, model, scoremodel, embed_model, preembed_model, image_embed_model, stagger=args.stagger, verbose=True, n_samp=20)
    odc.append(odc_)

print(get_mean_std_formatted(odc))

100%|███████████████████████████████████████████████████████████| 3040/3040 [02:27<00:00, 20.64it/s]

86.1300±0.2856


  5%|██▊                                                         | 101/2135 [00:26<08:46,  3.86it/s]


79.91499479690401

86.2855 with nsamp=20
86.3522 with nsamp=50
86.0198 with nsamp=100

50.0754 with nsamp=20
50.0065 with nsamp=50
50.4554 with nsamp=100

In [4]:
@torch.no_grad()
def compute_metrics(dataset, model, scoremodel, embed_model, preembed_model, image_embed_model=None, stagger=2, verbose=False, aggregator=None):

    loader = dataset.get_dataloader(batch_size=150, shuffle=True)

    C = embed_full_corpus(dataset, embed_model, preembed_model, image_embed_model=image_embed_model, aggregator=aggregator)
    # C is (N, n, xoutdim)

    netscores = []
    true_labels = []
    for n, (q, l) in enumerate(tqdm(loader, leave=False, disable=not verbose)):
        q = q.to(next(model.parameters()).device) 
        # q is bmd, c is Nnd, l is bN
        q = q + batch_get_white_noise(q, args.SNR)     # torch.randn_like(q) * noise
        q = embed_if_image_and_normalize(q, image_embed_model)
        q = embed_model(preembed_model(q))
        q = normalize(q)
        # q is (b, m, xoutdim)
        if aggregator is None:
            if not NOLAMMODEL:
                qct = torch.einsum("bmd,Nnd->bNmn", q, C)   # verified
                model_inputs = stagger_and_concat(qct, num_stagger=stagger)
                if stagger==0: 
                    model_inputs = model_inputs.squeeze(1)
                lambdas = torch.stack([model(x) for x in model_inputs])
                
                F_mat = Rm_mat.T @ (2*qct + (a_vec @ lambdas.transpose(2,3) @ A_mat).transpose(2,3))
                del qct
            else:
                F_mat = Rm_mat.T @ (
                    torch.stack([-(q[i][None].unsqueeze(2) - C.unsqueeze(1)).relu().sum(-1) for i in range(len(q))])
                )
                lambdas = torch.ones((len(q), len(C), M, 1), device=F_mat.device)

            P = gumbel_sinkhorn(F_mat, CFG.tau, CFG.n_sink_iter, noise=False)

            RmPC = Rm_mat @ P @ C.squeeze(-1)

            # if args.no_lamrelu:
            #     lamscore = lamwt * (lambdas.transpose(2,3) @ (b-A_mat @ Rm_mat @ P @ a_vec)).squeeze()
            # else:
            lamscore = lamwt * (lambdas.transpose(2,3) @ F.relu(b-A_mat @ Rm_mat @ P @ a_vec)).squeeze()
            normscore = torch.norm(q.unsqueeze(1) - RmPC, dim=[-1,-2])

            allscores = torch.stack([lamscore, normscore], dim=2)
            netscore = 2*scoremodel(-allscores).squeeze()      # b
            del P, F_mat, RmPC, q

        else:
            q = aggregator[0](q)
            # q is bd, C is Nd, we want bN scores
            # b1d - 1Nd = bNd --> sum across last dim to get bN scores
            netscore = 2 * F.sigmoid(-F.relu(q.unsqueeze(1) - C.unsqueeze(0)).sum(dim=-1))    # bN
            # TODO: fix this
            if args.deepset_mode == "cosine":     # 3
                netscore = 0.5 * (F.cosine_similarity(q.unsqueeze(1), C.unsqueeze(0), dim=-1) + 1)

        netscores.append(netscore.to('cpu'))
        true_labels.append(l.to('cpu'))

    netscores = torch.vstack(netscores)
    true_labels = torch.vstack(true_labels).squeeze() # 

    ranking = netscores.argsort(dim=1, descending=True)
    ranked_output = torch.gather(true_labels, dim=1, index=ranking)

    MRR = (1 / (ranked_output.argmax(dim=1) + 1)).mean().item()

    MAP = (torch.cumsum(ranked_output, dim=1) * ranked_output).float()
    MAP /= (torch.arange(ranked_output.shape[1]) + 1)
    MAP /= torch.sum(ranked_output, dim=1, keepdim=True)
    MAP = MAP.sum(dim=1).mean().item()

    return MAP, MRR

In [5]:
compute_metrics(test_dataset, model, scoremodel, embed_model, preembed_model, image_embed_model, stagger=args.stagger, verbose=True)

(0.9984142184257507, 0.998783528804779)

In [1]:
from run_inference import *

args = argparse.Namespace(expt_id="A20031054", device=4, old=False, dataset="audio")

experiment_id = args.expt_id
folder = "models" if not args.old else "models_old"
expt_root = f"{folder}/{experiment_id}/"

dataset = args.dataset
DEVICE = f"cuda:{args.device}" if torch.cuda.is_available() and args.device != -1 else "cpu"

import pickle
with open(os.path.join(expt_root, "args.pkl"), "rb") as file:
    args = pickle.load(file)

model, scoremodel, embed_model, preembed_model, aggregator = load_models(name="best", expt_root=expt_root, device=DEVICE, args=args)
model.eval(), scoremodel.eval(), embed_model.eval(), preembed_model.eval()
if aggregator is not None:
    aggregator.eval()

nwt, lamwt = next(scoremodel.parameters()).exp().detach().cpu()
nwt, lamwt = nwt.item(), lamwt.item()

print(f"normscore weight: {nwt:.4f}, lamscore weight: {lamwt:.4f}")

DATA_ROOT = f"final_data/{dataset}"
TRAIN_FILE = f"{DATA_ROOT}/dataset_train.hdf5"
VAL_FILE = f"{DATA_ROOT}/dataset_val.hdf5"
TEST_FILE = f"{DATA_ROOT}/dataset_test.hdf5"

image_embed_model, TRAIN_FILE, VAL_FILE, TEST_FILE = get_image_embed_model(dataset, [TRAIN_FILE, VAL_FILE, TEST_FILE], device=DEVICE)

train_dataset = PairDatasetTrain(TRAIN_FILE, num_q=args.num_q, negative_exploration=args.neg_expl)
val_dataset = PairDatasetTrain(VAL_FILE, num_q=args.num_q, negative_exploration=args.neg_expl)
test_dataset = PairDatasetTest(TEST_FILE)

##### HPARAMS ######
b = args.b          # hinge margin for negative gap penalty (b-Apa)
b1 = args.b1         # hinge margin for positive gap penalty (Apa-b)
delta = args.delta     # hinge margin for contrastive loss
lr = args.lr
nepochs = args.nepochs
stagger = args.stagger
lamwt = args.lamwt    # loss coefficient for negative gap penalty
gapwt = args.gapwt       # loss coefficient for positive gap penalty
M = test_dataset.q[0].shape[0]
N = test_dataset.c[0].shape[0]
####################

A_mat, a_vec, Rm_mat = get_opas_constants(M, N, DEVICE)

CFG = AttributeDict({
    'tau': 1,
    'n_sink_iter': 20,
    'n_samples': 1,
})

DEEPSET = args.deepset
NOLAMMODEL = args.no_lammodel   
positive_samples = 10

Loading `best` model
normscore weight: 0.5910, lamscore weight: 0.0740


In [60]:
trainloader = train_dataset.get_dataloader(batch_size=args.batch_size, shuffle=True)
valloader = val_dataset.get_dataloader(batch_size=args.batch_size, shuffle=False)

with torch.no_grad():
    for n, (q, c, l) in enumerate(valloader):
        qpos, cpos, lpos = train_dataset.get_positive_samples(positive_samples)
        q = torch.cat((q, qpos), dim=0).to(DEVICE)
        c = torch.cat((c, cpos), dim=0).to(DEVICE)
        l = torch.cat((l, lpos), dim=0)
        if args.enforce_order:
            qneg = []   # shuffled
            cneg = []   # repeats
            lneg = []   # 0
            for q_,c_,l_ in zip(qpos,cpos,lpos):
                for _ in range(5):
                    qneg.append(q_[torch.randperm(q_.shape[0])])
                    cneg.append(c_)
                    lneg.append(torch.zeros_like(l_))
            qneg, cneg, lneg = torch.stack(qneg).to(DEVICE), torch.stack(cneg).to(DEVICE), torch.stack(lneg)
            q = torch.cat((q, qneg), dim=0)
            c = torch.cat((c, cneg), dim=0)
            l = torch.cat((l, lneg), dim=0)

        q = q + batch_get_white_noise(q, args.SNR)
        q, c = embed_if_image_and_normalize(q, image_embed_model), embed_if_image_and_normalize(c, image_embed_model)

        if args.use_sing_xfmer:
            qc = torch.cat((q, c), dim=1)
            qc = embed_model(preembed_model(qc))
            q, c = qc[:, :M], qc[:, M:]
            q, c = normalize(q, c)
        else:
            # main forward pass, !! Change this in compute_metrics as well !!
            q, c = embed_model(preembed_model(q)), embed_model(preembed_model(c))
            q, c = normalize(q, c)

        # import ipdb; ipdb.set_trace()

        if not DEEPSET:
            if not NOLAMMODEL:
                qct = torch.einsum("bmd,bnd->bmn", q, c)
                
                if args.use_linear_lammodel:
                    lambdas = model(torch.cat((q, c), dim=1).flatten(start_dim=1)).unsqueeze(-1)
                else:
                    model_inputs = stagger_and_concat(qct, num_stagger=stagger)
                    if stagger==0: 
                        model_inputs = model_inputs.squeeze(1)
                    lambdas = model(model_inputs)

                F_mat = Rm_mat.T @ (2*qct + internal_lamwt * (a_vec @ lambdas.transpose(1,2) @ A_mat).transpose(1,2))
            else:
                F_mat = Rm_mat.T @ -(q.unsqueeze(2) - c.unsqueeze(1)).relu().sum(-1)
                lambdas = torch.ones((q.shape[0], M, 1), device=DEVICE)
            
            if args.single_step_norm == 1:
                P = F_mat / F_mat.sum(dim=-2, keepdims=True)    # normalizing on the row dimension
            elif args.single_step_norm == 2:
                P = F_mat.exp() / F_mat.exp().sum(dim=-2, keepdims=True)
            else:
                P = gumbel_sinkhorn(F_mat, CFG.tau, CFG.n_sink_iter, noise=False)

            # RmPC = torch.einsum("mn,bnn,bnd->bmd", Rm_mat, P, c)  # sanity fail
            RmPC = Rm_mat @ torch.bmm(P, c)

            if args.no_lamrelu:
                lamscore = lamwt * (lambdas.transpose(1,2) @ (b-A_mat @ Rm_mat @ P @ a_vec)).squeeze()
            else:
                lamscore = lamwt * (lambdas.transpose(1,2) @ F.relu(b-A_mat @ Rm_mat @ P @ a_vec)).squeeze()
            normscore = torch.norm(q - RmPC, dim=[1,2])

            allscores = torch.stack([lamscore, normscore], dim=1)
            netscore = 2*scoremodel(-allscores).squeeze()
        else:
            q, c = aggregator[0](q), aggregator[1](c)
            if args.deepset_mode == "normalized":   # 2
                netscore = 2 * F.sigmoid(-F.relu(q - c).sum(dim=-1))    # normalized to 0-1, 0 for worst, 1 for best (==0 loss)
            elif args.deepset_mode == "base":       # 1
                netscore = -F.relu(q - c).sum(dim=-1)                   # un-normalized scores
            elif args.deepset_mode == "cosine":     # 3
                netscore = 0.5 * (F.cosine_similarity(q, c, dim=-1) + 1)

        pos_score = torch.atleast_1d(netscore[torch.where(l==1)])
        neg_score = netscore[torch.where(l==0)]
        
        neg_minus_pos = (neg_score.unsqueeze(0) - pos_score.unsqueeze(1)).reshape(-1)   # dim(pos_score) * dim(neg_score)

        loss = F.relu(delta + neg_minus_pos).mean() 
        if gapwt > 0 and not DEEPSET:
            loss += gapwt * F.relu(A_mat @ Rm_mat @ P @ a_vec - b1).mean()
        
        print(f"{loss.item():.6f}")
        if n>5:
            break

0.139400
0.103757
0.134052
0.118913
0.136689
0.160580
0.090042


In [61]:
pos_score.mean(), neg_score.mean(), (neg_minus_pos + delta).mean(), loss

(tensor(0.7302, device='cuda:4'),
 tensor(0.4475, device='cuda:4'),
 tensor(0.0173, device='cuda:4'),
 tensor(0.0900, device='cuda:4'))

In [62]:
from opas.data import DummyDataset

aggregator_flipped = [aggregator[1], aggregator[0]]

test_query_embeds = embed_full_corpus(DummyDataset(test_dataset.q), embed_model, preembed_model, image_embed_model, inner_batch_size=400, aggregator=aggregator_flipped, verbose=True)
test_corpus_embeds = embed_full_corpus(DummyDataset(test_dataset.c), embed_model, preembed_model, image_embed_model, inner_batch_size=400, aggregator=aggregator, verbose=True)

In [ ]:
compute_metrics(train_dataset, )

In [63]:
test_query_embeds = test_query_embeds.cpu()
test_corpus_embeds = test_corpus_embeds.cpu()

In [64]:
cs = ((test_query_embeds @ test_corpus_embeds.T) / (test_query_embeds.norm(dim=-1).unsqueeze(-1) @ test_corpus_embeds.norm(dim=-1).unsqueeze(0)) + 1) / 2
cs 

tensor([[0.4672, 0.4678, 0.4630,  ..., 0.4072, 0.4063, 0.4072],
        [0.4834, 0.4846, 0.4879,  ..., 0.4281, 0.4279, 0.4295],
        [0.4464, 0.4481, 0.4469,  ..., 0.4076, 0.4081, 0.4082],
        ...,
        [0.4189, 0.4189, 0.4158,  ..., 0.4209, 0.4217, 0.4214],
        [0.4551, 0.4551, 0.4488,  ..., 0.4281, 0.4298, 0.4288],
        [0.4176, 0.4186, 0.4158,  ..., 0.4261, 0.4259, 0.4258]])

In [75]:
l = torch.stack([test_dataset[i][1] for i in range(50)])
l.shape

torch.Size([50, 4864])

In [76]:
cs[0]

tensor([0.4672, 0.4678, 0.4630,  ..., 0.4072, 0.4063, 0.4072])

In [85]:
labels = l
labels[0]

from sklearn.metrics import average_precision_score

np.mean([average_precision_score(labels[i], cs[i]) for i in range(1)])

a,b = torch.sort(cs[0], descending=True)

b

tensor([1772, 1773, 1765,  ..., 4265, 4270, 4259])

In [ ]:
ranking = torch.argsort(cs, dim=1, descending=True)
lranked = torch.gather(l, dim=1, index=ranking[:10])

ranked_output = lranked
MAP = (torch.cumsum(ranked_output, dim=1) * ranked_output).float()
MAP /= (torch.arange(ranked_output.shape[1]) + 1)
MAP /= torch.sum(ranked_output, dim=1, keepdim=True)
MAP = MAP.sum(dim=1).mean().item()
MAP

0.004065193701535463

In [90]:
tst = PairDatasetTest(TRAIN_FILE)

In [92]:
proj = torch.ones((13, 4))
F.pad(proj, (0, 20-proj.shape[-1]), value=0).shape

torch.Size([13, 20])

In [ ]:



F.pad(proj.values, (0, self.outdim - proj.values.shape[-1]), value=0)
